# Evaluation of ML Algorithms

In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import numpy as np

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

lr = LogisticRegression(max_iter=10000, random_state=0).fit(X_train, y_train)
y_pred = lr.predict(X_test)


## Accuracy, Precision, Recall, F1-score

In [2]:
def accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

def precision(y_true, y_pred, threshold=0.5):
    TP = np.sum(y_true & (y_pred >= threshold))
    FP = np.sum((y_true == 0) & (y_pred >= threshold))
    return TP / (TP + FP) if (TP + FP) > 0 else 1

def recall(y_true, y_pred, threshold=0.5):
    TP = np.sum(y_true & (y_pred >= threshold))
    FN = np.sum(y_true & (y_pred < threshold))
    return TP / (TP + FN) if (TP + FN) > 0 else 1

def f1(y_true, y_pred):
    P = precision(y_true, y_pred)
    R = recall(y_true, y_pred)
    return 2*(P * R) / (P + R)

In [3]:
def print_metrics(metrics, y_true, y_pred):
    print(f"{'SKLearn Metrics':^30s}{'My Metrics':^30s}")
    print(f"{'Metric':20s}{'Score':10s}{'|':3s}{'Metric':20s}{'Score':20s}")
    for sklearnMetric, myMetric in metrics:
        print(f"{sklearnMetric.__name__:20s}{sklearnMetric(y_true, y_pred):<10f}{'|':3s}{myMetric.__name__:20s}{myMetric(y_true, y_pred):<20f}")

In [4]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, precision_recall_curve, roc_auc_score

metrics = [
    (accuracy_score, accuracy),
    (precision_score, precision),
    (recall_score, recall),
    (f1_score, f1)
]
print_metrics(metrics, y_test, y_pred)

       SKLearn Metrics                  My Metrics          
Metric              Score     |  Metric              Score               
accuracy_score      0.956140  |  accuracy            0.956140            
precision_score     0.945946  |  precision           0.945946            
recall_score        0.985915  |  recall              0.985915            
f1_score            0.965517  |  f1                  0.965517            


## AUC of the ROC curve

In [5]:
from sklearn.metrics import roc_auc_score

def trapz(y, x):
    return np.sum([0.5 * (y[i + 1] + y[i]) * (x[i + 1] - x[i]) for i in range(len(x) - 1)])

def fpr(y_true, y_prob, threshold):
    FP = np.sum((y_true == 0) & (y_prob < threshold))
    TN = np.sum((y_true == 0) & (y_prob >= threshold))
    return FP / (FP + TN) if (FP + TN) > 0 else 1

def roc_auc(y_true, y_prob):
    FPR, TPR = [], []

    for threshold in np.arange(0.0, 1.01, .01):
        FPR.append(fpr(y_true, y_prob, threshold))
        TPR.append(recall(y_true, y_prob, threshold))

    return trapz(TPR, FPR)

print("AUC of the ROC curve: " + str(roc_auc(y_test, lr.predict_proba(X_test)[:, 1])))
print("SKLearn roc_auc_score: " + str(roc_auc_score(y_test, lr.decision_function(X_test))))

AUC of the ROC curve: 0.9977071732721913
SKLearn roc_auc_score: 0.9977071732721913


## AUC of the Precision-Recall Curve

In [6]:
from sklearn.metrics import precision_recall_curve, auc

def precision_recall_auc(y_true, y_prob):
    precisions, recalls = [], []

    for threshold in np.arange(0.0, 1.01, .01):
        precisions.append(precision(y_true, y_prob, threshold))
        recalls.append(recall(y_true, y_prob, threshold))

    return -1 * trapz(precisions, recalls)

prec, rec, _ = precision_recall_curve(y_test, lr.decision_function(X_test))

print("AUC of the Precision-Recall curve: " + str(precision_recall_auc(y_test, lr.predict_proba(X_test)[:, 1])))
print("SKLearn precision_recall_curve: " + str(auc(rec, prec)))

AUC of the Precision-Recall curve: 0.9986732187872313
SKLearn precision_recall_curve: 0.9986732187872311
